In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset , DataLoader
import torchvision
from sklearn.metrics import accuracy_score, confusion_matrix

In [15]:
#Dataset and Dataloader
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
trainset = torchvision.datasets.MNIST(root='./data',train=True,download=True,transform=transform)
testset = torchvision.datasets.MNIST(root='./data',train=False,download=True,transform=transform)

In [16]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64,shuffle=True)

In [22]:
# Building CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
        # Convolutional layer
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
      # Fully Connected Layers
        self.fc_layers = nn.Sequential(
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        #for feature extraction
        x = self.conv_layers(x)
        # for flattening
        x = x.view(x.size(0),-1)
        # final classification
        x = self.fc_layers(x)
        return x

In [23]:
model = CNN()
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=3136, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [24]:
criterion = nn.CrossEntropyLoss()          
optimizer = optim.Adam(model.parameters())

In [26]:
# train the CNN
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for Xb,yb in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(Xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    print(f"epoch={epoch+1}/{epochs}  & loss = {epoch_training_loss/len(trainloader)}")

epoch=1/10  & loss = 0.13716629128218064
epoch=2/10  & loss = 0.04187914953124759
epoch=3/10  & loss = 0.027777528192146496
epoch=4/10  & loss = 0.020014307571438898
epoch=5/10  & loss = 0.01710108191132262
epoch=6/10  & loss = 0.01271365662379665
epoch=7/10  & loss = 0.011503741094494122
epoch=8/10  & loss = 0.008674906883847791
epoch=9/10  & loss = 0.0066752029668114
epoch=10/10  & loss = 0.0065347693721818124


In [ ]:
# evalution
correct = 0
total = 0
all_pred = []
all_target = []
model.eval()
with torch.no_grad():
    for Xb,yb in testloader:
        outputs = model.forward(Xb)
        _, predicted = torch.max(outputs,1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

        all_pred.extend(predicted.cpu().numpy())
        all_target.extend(yb.cpu().numpy())
cnn_Accuracy = correct/total*100
print(cnn_Accuracy)
cnn_cm = confusion_matrix(all_pred,all_target)
print(cnn_cm)

In [ ]:
# RNN model
class RNN(nn.Module):
    def __init__(self, input_size,hidden_size =128,num_layers = 1):
        super().__init__()

        self.hidden_layer = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fully connected layer 
        self.fc = nn.Linear(hidden_size,10)

    def forward(self,x):
        # hidden layer of all timestamp
        if x.dim() == 4:
            x = x.squeeze(1)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_layer)
        out, _ = self.rnn(x,h0)
        out = self.fc(out[:,-1,:])
        return out

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rnn_model = RNN(input_size=28,hidden_size=128,num_layers=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(),lr=0.001)

In [ ]:
epochs = 5
for epoch in range(rnn_epochs):
    rnn_model.train()
    epoch_loss = 0.0

    for Xb,yb in trainloader:
        Xb,yb = Xb.to(device),yb.to(device)
        optimizer.zero_grad()
        outputs = rnn_model(Xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step()

        epoch_loss +=loss.item()
    print(f"epoch = [{epoch+1}/{epochs}] & loss = {epoch_loss/len(trainloader)}")

In [ ]:
#evalution
rnn_preds = []
rnn_target = []
rnn_model.eval()
with torch.no_grad():
    for Xb, yb in testloader:
        Xb,yb = Xb.to(device),yb.to(device)
        outputs = rnn_model(Xb)
        _, predicted = torch.max(outputs,1)

        rnn_preds.extend(predicted.cpu().numpy())
        rnn_target.extend(yb.cpu().numpy())
        
# final accuracy 
accuracy = accuracy_score(rnn_target,rnn_preds)
print(f"RNN TEST Accuracy={accuracy*100}\nCOnfusion Matrix")
rnn_cm = confusion_matrix(rnn_target,rnn_preds)
print(rnn_cm)

In [ ]:
print("CNN Accuracy :", cnn_accuracy*100)
print("RNN Accuracy :", rnn_accuracy*100)